# Solar Disk Center Model Training

Train and evaluate a CNN-based solar-disk localisation model.

**Configuration:** Dataset and model-output paths must be configured locally.

In [1]:
# Step 1: Import dependencies and initialise this stage.
import os
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda-12.3/targets/x86_64-linux/lib/'

In [ ]:
# Step 2: Import dependencies and initialise this stage.
import tensorflow as tf
if tf.test.gpu_device_name():
    print('Default GPU Device: {}'.format(tf.test.gpu_device_name()))
else:
    print("Please install GPU version of TF")

In [ ]:
# Step 3: Import dependencies and initialise this stage.
import keras
import cv2
from PIL import Image
import numpy as np
from matplotlib import pyplot as plt
import tifffile as tiff
import time

In [ ]:
# Step 4: Import dependencies and initialise this stage.
import tensorflow as tf
tf.test.is_built_with_cuda() # To test CUDA support for your Tensorflow installation
tf.config.list_physical_devices('GPU')
tf.test.is_gpu_available(cuda_only=False, min_cuda_compute_capability=None) # To confirm that the GPU is available to Tensorflow
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

In [ ]:
# Step 5: Run the next processing stage.
image_directory = '/media/storage/development/aditya_detection/'
mask_directory = '/media/storage/development/aditya_detection_mask/'

In [ ]:
# Step 6: Run the next processing stage.
image_dataset = []   
mask_dataset = []

In [ ]:
# save the images in array
images = sorted(os.listdir(image_directory))
for i, image_name in enumerate(images):    
    if (image_name.split('.')[1] == 'tif'):
        image = cv2.imread(image_directory+image_name,cv2.COLOR_BGR2RGB)
        image = Image.fromarray(image)
        image = image.resize((256,256)) #491,512#480,512
        image_dataset.append(np.array(image))


# save the masks in array
masks = sorted(os.listdir(mask_directory))
for i, image_name in enumerate(masks):
    if (image_name.split('.')[1] == 'tif'):
        image = cv2.imread(mask_directory+image_name, 0)
        image = Image.fromarray(image)
        image = image.resize((256,256))
        mask_dataset.append(np.array(image))


In [ ]:
# apply the erosion and dilation operator on masks 
images = np.array(image_dataset)
masks_ = np.array(mask_dataset)
kernel = np.ones((1,1), np.uint8)
erosion = cv2.erode(masks_, kernel, iterations=1)
bg = cv2.dilate(masks_, kernel, iterations = 1)
masks = np.expand_dims(bg, -1)

In [ ]:
#Define the model
%env SM_FRAMEWORK=tf.keras
import segmentation_models as sm

In [ ]:
# Step 10: Run the next processing stage.
BACKBONE = 'resnet50'
preprocess_input1 = sm.get_preprocessing(BACKBONE)

In [ ]:
# Step 11: Run the next processing stage.
images1=preprocess_input1(images)

In [ ]:
# Step 12: Import dependencies and initialise this stage.
import random
import numpy as np
image_number = random.randint(0, len(images1))
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(images1, masks, test_size = 0.2, random_state = 42)

In [ ]:
#New generator with rotation and shear where interpolation that comes with rotation and shear are thresholded in masks. 
#This gives a binary mask rather than a mask with interpolated values. 
seed=24
from keras.preprocessing.image import ImageDataGenerator

img_data_gen_args = dict(rotation_range=90,
                     width_shift_range=0.3,
                     height_shift_range=0.3,
                     shear_range=0.5,
                     zoom_range=0.3,
                     horizontal_flip=True,
                     vertical_flip=True,
                     fill_mode='reflect')

mask_data_gen_args = dict(rotation_range=90,
                     width_shift_range=0.3,
                     height_shift_range=0.3,
                     shear_range=0.5,
                     zoom_range=0.3,
                     horizontal_flip=True,
                     vertical_flip=True,
                     fill_mode='reflect',
                     preprocessing_function = lambda x: np.where(x>0, 1, 0).astype(x.dtype)) #Binarize the output again. 

image_data_generator = ImageDataGenerator(**img_data_gen_args)
image_data_generator.fit(X_train, augment=True, seed=seed)

image_generator = image_data_generator.flow(X_train, seed=seed)
valid_img_generator = image_data_generator.flow(X_test, seed=seed)


mask_data_generator = ImageDataGenerator(**mask_data_gen_args)
mask_data_generator.fit(y_train, augment=True, seed=seed)
mask_generator = mask_data_generator.flow(y_train, seed=seed)
valid_mask_generator = mask_data_generator.flow(y_test, seed=seed)


def my_image_mask_generator(image_generator, mask_generator):
    train_generator = zip(image_generator, mask_generator)
    for (img, mask) in train_generator:
        yield (img, mask)
my_generator = my_image_mask_generator(image_generator, mask_generator)

validation_datagen = my_image_mask_generator(valid_img_generator,valid_mask_generator)

In [ ]:
# define model
model = sm.Unet(BACKBONE, encoder_weights='imagenet',encoder_freeze=True)
model.compile('Adam', loss=sm.losses.bce_jaccard_loss, metrics=[sm.metrics.iou_score])
print(model.summary())

In [ ]:
# Fit the model
history = model.fit(my_generator, validation_data=validation_datagen,steps_per_epoch=183,validation_steps=100,batch_size=32, epochs=60,shuffle=True)
model.save('/home/dibya/limb_detection.hdf5')

In [ ]:
#plot the training and validation loss at each epoch
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'y', label='Training loss')
plt.plot(epochs, val_loss, 'r', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# Step 17: Run the next processing stage.
acc = history.history['iou_score']
val_acc = history.history['val_iou_score']
print(val_acc)

In [ ]:
# plot the training and validation accuracy at each epoch
plt.plot(epochs, acc, 'y', label='Training IOU')
plt.plot(epochs, val_acc, 'r', label='Validation IOU')
plt.title('Training and validation IOU')
plt.xlabel('Epochs')
plt.ylabel('IOU')
plt.ylim(0,1)
plt.legend()
plt.show()

In [ ]:
#IOU
y_pred=model.predict(X_test)
y_pred_thresholded = y_pred > 0.5

intersection = np.logical_and(y_test, y_pred_thresholded)
union = np.logical_or(y_test, y_pred_thresholded)
iou_score = np.sum(intersection) / np.sum(union)
print("IoU socre is: ", iou_score)